# **Building basic CNN model using Pytorch**

# import the necessary packages

In [5]:
from torch.nn import (
    Module ,
    Conv2d,
    Linear,
    MaxPool2d,
    ReLU,
    LogSoftmax)
from torch import flatten

# Defining CNN model architecture in LeNet class in pytorch

In [6]:
class LeNet(Module):
  def __init__(self, numChannels, classes):
    super(LeNet, self).__init__()
    self.conv1 = Conv2d(in_channels = numChannels, out_channels =20, kernel_size =(5,5))
    self.relu1 = ReLU()
    self.maxpool1 = MaxPool2d(kernel_size = (2,2), stride = (2,2))

    self.conv2 = Conv2d(in_channels=20, out_channels=50, kernel_size=(5,5))
    self.relu2 = ReLU()
    self.maxpool2 = MaxPool2d(kernel_size = (2,2), stride = (2,2))

    self.fc1 = Linear(in_features = 800, out_features =500)
    self.relu3 = ReLU()

    self.fc2 = Linear(in_features=500, out_features= classes)
    self.LogSoftmax = LogSoftmax(dim=1)

  def forward(self, x):
    x = self.conv1(x)
    x = self.relu1(x)
    x = self.maxpool1(x)

    x = self.conv2(x)
    x = self.relu2(x)
    x = self.maxpool2(x)

    x = flatten(x,1)
    x = self.fc1(x)
    x = self.relu3(x)

    x = self.fc2(x)
    output = self.LogSoftmax(x)

    return output

In [26]:
# set the matplotlib backend so figures can be saved in the background
import matplotlib
matplotlib.use("Agg")

#import the necessary packages
from sklearn.metrics import classification_report
from torch.utils.data import random_split, DataLoader
from torchvision.transforms import ToTensor
from torchvision.datasets import KMNIST
from torch.optim import Adam
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
import argparse
import torch
import time

# Define training hyperparameters

In [28]:
INIT_LR = 1e-3
BATCH_SIZE = 64
EPOCHS = 10


# Define the train and val splits

In [27]:
TRAIN_SPLIT = 0.75
VAL_SPLIT = 1 - TRAIN_SPLIT
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the KMNIST dataset

In [10]:
print("[INFO] loading the KMNIST dataset...")
trainData = KMNIST(root="data", train=True, download=True,
	transform=ToTensor())
testData = KMNIST(root="data", train=False, download=True,
	transform=ToTensor())

print("[INFO] generating the train/validation split...")
numTrainSamples = int(len(trainData) * TRAIN_SPLIT)
numValSamples = int(len(trainData) * VAL_SPLIT)
(trainData, valData) = random_split(trainData,
	[numTrainSamples, numValSamples],
	generator=torch.Generator().manual_seed(42))

[INFO] loading the KMNIST dataset...


100%|██████████| 18.2M/18.2M [00:18<00:00, 968kB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 186kB/s]
100%|██████████| 3.04M/3.04M [00:03<00:00, 1.01MB/s]
100%|██████████| 5.12k/5.12k [00:00<00:00, 6.68MB/s]

[INFO] generating the train/validation split...


# Initialize the train, validation, and test data loaders

In [11]:
trainDataLoader = DataLoader(trainData, shuffle=True , batch_size = BATCH_SIZE)
valDataLoader = DataLoader(valData, batch_size=BATCH_SIZE)
testDataLoader = DataLoader(testData, batch_size = BATCH_SIZE)

trainSteps = len(trainDataLoader.dataset) // BATCH_SIZE
valSetps = len(valDataLoader.dataset) // BATCH_SIZE

# Initialize the LeNet model

In [12]:
model = LeNet(
    numChannels=1,
    classes = len(trainData.dataset.classes)
)
opt = Adam(model.parameters(), lr=INIT_LR)
lossFn = nn.NLLLoss()

# initialize a dictionary to store training history
H = {
    "train_loss":[],
    "train_acc":[],
    "val_loss":[],
    "val_acc":[]
}



In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training first Convolutional Neural Network

In [14]:
for e in range(0, EPOCHS):

	model.train()
	totalTrainLoss = 0
	totalValLoss = 0
	trainCorrect = 0
	valCorrect = 0
	for (x, y) in trainDataLoader:
		(x, y) = (x.to(device), y.to(device))
		pred = model(x)
		loss = lossFn(pred, y)
		opt.zero_grad()
		loss.backward()
		opt.step()
		totalTrainLoss += loss
		trainCorrect += (pred.argmax(1) == y).type(
			torch.float).sum().item()

# Switch off autograd for evaluation and evalute the LeNet Model

In [19]:
with torch.no_grad():
  model.eval()
  preds = []
  for (x,y) in testDataLoader:
    x = x.to(device)
    pred = model(x)
    preds.extend(pred.argmax(axis=1).cpu().numpy())

print(classification_report(testData.targets.cpu().numpy(),
                            np.array(preds),
                            target_names = testData.classes))


              precision    recall  f1-score   support

           o       0.97      0.90      0.93      1000
          ki       0.96      0.94      0.95      1000
          su       0.92      0.93      0.93      1000
         tsu       0.97      0.96      0.97      1000
          na       0.92      0.94      0.93      1000
          ha       0.94      0.94      0.94      1000
          ma       0.94      0.96      0.95      1000
          ya       0.94      0.96      0.95      1000
          re       0.96      0.96      0.96      1000
          wo       0.95      0.96      0.96      1000

    accuracy                           0.95     10000
   macro avg       0.95      0.95      0.95     10000
weighted avg       0.95      0.95      0.95     10000



In [25]:
# plot the training loss and accuracy

plt.figure()
plt.plot(H["train_loss"], label="train_loss")
plt.plot(H["val_loss"], label="val_loss")
plt.plot(H["train_acc"], label="train_acc")
plt.plot(H["val_acc"], label="val_acc")
plt.title("Training Loss and Accuracy on Dataset")
plt.xlabel("Epoch #")
plt.ylabel("Loss/Accuracy")
plt.legend(loc="lower left")
plt.show()


In [ ]:
# serialize the model to disk
torch.save(model, args["model"])